# Bank Marketing: Model Comparison

## Objective

This notebook compares several machine learning models using cross-validation on the training data.

The goals are to:

- compare multiple suitable classification models
- avoid using the test set during model selection
- evaluate models using metrics suitable for class imbalance
- identify the strongest candidates for hyperparameter tuning

The untouched test set will only be used after the final model has been selected.

In [1]:
from ucimlrepo import fetch_ucirepo

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features.copy()
y = bank_marketing.data.targets["y"].map({
    "no": 0,
    "yes": 1
})

X = X.drop(columns=["duration"])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numerical_features = (
    X_train
    .select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

Training shape: (36168, 15)
Test shape: (9043, 15)


## Preprocessing pipeline

The same preprocessing strategy from the previous notebook is reused inside every model pipeline.

This ensures that preprocessing is fitted separately inside each cross-validation training fold and prevents data leakage.

In [2]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="missing"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

## Candidate models

Four actual machine learning models are compared:

- Logistic regression as an interpretable linear baseline
- Decision tree as a simple non-linear model
- Random forest as a bagging ensemble
- Histogram gradient boosting as a boosting model

The dummy classifier is retained only as the minimum benchmark.

In [3]:
models = {
    "Dummy classifier": DummyClassifier(
        strategy="most_frequent"
    ),

    "Logistic regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),

    "Decision tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "Random forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Histogram gradient boosting": HistGradientBoostingClassifier(
        random_state=42
    )
}

print("Models:")
for model_name in models:
    print("-", model_name)

Models:
- Dummy classifier
- Logistic regression
- Decision tree
- Random forest
- Histogram gradient boosting


## Cross-validation strategy

Stratified five-fold cross-validation is used to compare the candidate models.

Stratification preserves the class distribution inside every fold. Each model is evaluated on the same folds using several metrics suited to the imbalanced target.

The primary comparison metrics are:

- recall
- F1-score
- ROC-AUC
- PR-AUC

Accuracy and precision are also recorded, but they are not used alone to select the best model.

In [4]:
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

## Cross-validated model comparison

Each classifier is placed inside a complete pipeline containing preprocessing and model training.

Because preprocessing is fitted separately within each training fold, information from validation folds cannot influence imputation, scaling, or encoding.

In [5]:
model_results = []

for model_name, model in models.items():
    print(f"Evaluating: {model_name}")

    model_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", model)
        ]
    )

    scores = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=cross_validation,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )

    model_results.append({
        "model": model_name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "precision_mean": scores["test_precision"].mean(),
        "recall_mean": scores["test_recall"].mean(),
        "f1_mean": scores["test_f1"].mean(),
        "roc_auc_mean": scores["test_roc_auc"].mean(),
        "pr_auc_mean": scores["test_pr_auc"].mean(),
        "fit_time_mean": scores["fit_time"].mean()
    })

comparison_df = (
    pd.DataFrame(model_results)
    .set_index("model")
    .sort_values(
        by="pr_auc_mean",
        ascending=False
    )
)

display(comparison_df.round(4))

Evaluating: Dummy classifier
Evaluating: Logistic regression
Evaluating: Decision tree
Evaluating: Random forest
Evaluating: Histogram gradient boosting


,accuracy_mean,accuracy_std,precision_mean,recall_mean,f1_mean,roc_auc_mean,pr_auc_mean,fit_time_mean
model,,,,,,,,
Histogram gradient boosting,0.8935,0.0022,0.6216,0.2290,0.3345,0.7959,0.4425,3.9006
Random forest,0.8924,0.0014,0.6198,0.2080,0.3113,0.7804,0.4163,16.8912
Logistic regression,0.7550,0.0055,0.2660,0.6218,0.3726,0.7637,0.3985,0.6710
Decision tree,0.8356,0.0031,0.2995,0.3028,0.3011,0.6045,0.1724,1.0617
Dummy classifier,0.8830,0.0001,0.0000,0.0000,0.0000,0.5000,0.1170,0.2356


### Cross-validation interpretation

The candidate models show different trade-offs.

Histogram gradient boosting achieves the strongest overall ranking performance, with the highest ROC-AUC and PR-AUC. It also has relatively high precision, but its recall is low, meaning that it misses many actual subscribers.

Random forest behaves similarly but is slower and performs slightly worse than histogram gradient boosting on most metrics.

Logistic regression achieves the highest recall and F1-score. It identifies substantially more subscribers, but its lower precision indicates that it also generates more false positives.

The decision tree performs worse than the stronger candidate models and offers no clear advantage. The dummy classifier confirms that high accuracy can be achieved without identifying any subscribers.

Therefore, histogram gradient boosting and logistic regression will be retained for hyperparameter tuning. They represent two useful strategies:

- histogram gradient boosting for stronger ranking and precision
- logistic regression for stronger recall and minority-class detection

## Logistic regression hyperparameter tuning

Logistic regression is tuned using stratified cross-validation.

The search focuses on regularization strength and class weighting. PR-AUC is used as the primary optimization metric because the positive class is imbalanced and identifying likely subscribers is the main objective.

In [6]:
logistic_tuning_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

logistic_parameter_grid = {
    "classifier__C": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ],
    "classifier__class_weight": [
        None,
        "balanced"
    ],
    "classifier__solver": [
        "liblinear",
        "lbfgs"
    ]
}

logistic_search = RandomizedSearchCV(
    estimator=logistic_tuning_pipeline,
    param_distributions=logistic_parameter_grid,
    n_iter=12,
    scoring="average_precision",
    cv=cross_validation,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

logistic_search.fit(X_train, y_train)

print("Best parameters:")
print(logistic_search.best_params_)

print("\nBest cross-validated PR-AUC:")
print(round(logistic_search.best_score_, 4))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best parameters:
{'classifier__solver': 'liblinear', 'classifier__class_weight': None, 'classifier__C': 0.1}

Best cross-validated PR-AUC:
0.4025


### Logistic regression tuning result

The best logistic regression configuration used:

- regularization strength `C = 0.1`
- the `liblinear` solver
- no automatic class weighting

The best cross-validated PR-AUC was approximately 0.4025.

Although class weighting previously increased recall, the unweighted configuration achieved better probability-ranking performance according to PR-AUC. Classification threshold selection will later be considered separately because the default threshold of 0.5 may not provide the preferred precision-recall trade-off.

In [7]:
tuned_logistic_scores = cross_validate(
    estimator=logistic_search.best_estimator_,
    X=X_train,
    y=y_train,
    cv=cross_validation,
    scoring=scoring,
    n_jobs=-1
)

tuned_logistic_summary = pd.Series({
    metric: tuned_logistic_scores[f"test_{metric}"].mean()
    for metric in scoring
})

display(
    tuned_logistic_summary
    .to_frame("Tuned logistic regression")
    .round(4)
)

,Tuned logistic regression
accuracy,0.8927
precision,0.6643
recall,0.1676
f1,0.2675
roc_auc,0.7633
pr_auc,0.4025


## Histogram gradient boosting hyperparameter tuning

Histogram gradient boosting is tuned because it achieved the strongest overall ranking performance during the initial model comparison.

The search explores model complexity, learning rate, regularization, and minimum leaf size. PR-AUC remains the optimization metric.

In [8]:
histogram_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            HistGradientBoostingClassifier(
                random_state=42
            )
        )
    ]
)

histogram_parameter_grid = {
    "classifier__learning_rate": [
        0.03,
        0.05,
        0.1,
        0.2
    ],
    "classifier__max_iter": [
        100,
        200,
        300
    ],
    "classifier__max_leaf_nodes": [
        15,
        31,
        63
    ],
    "classifier__min_samples_leaf": [
        10,
        20,
        40
    ],
    "classifier__l2_regularization": [
        0.0,
        0.1,
        1.0,
        5.0
    ]
}

histogram_search = RandomizedSearchCV(
    estimator=histogram_pipeline,
    param_distributions=histogram_parameter_grid,
    n_iter=20,
    scoring="average_precision",
    cv=cross_validation,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

histogram_search.fit(X_train, y_train)

print("Best parameters:")
print(histogram_search.best_params_)

print("\nBest cross-validated PR-AUC:")
print(round(histogram_search.best_score_, 4))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters:
{'classifier__min_samples_leaf': 40, 'classifier__max_leaf_nodes': 31, 'classifier__max_iter': 100, 'classifier__learning_rate': 0.1, 'classifier__l2_regularization': 1.0}

Best cross-validated PR-AUC:
0.4488


### Hyperparameter tuning interpretation

The tuned logistic regression achieved a cross-validated PR-AUC of approximately 0.4025.

Its precision increased substantially, but its recall decreased to approximately 16.8%. This means that the tuned model makes fewer positive predictions and is more accurate when it predicts subscription, but it misses most actual subscribers.

The tuned histogram gradient boosting model achieved a higher cross-validated PR-AUC of approximately 0.4488. This indicates stronger overall ability to rank likely subscribers in the imbalanced dataset.

Based on the primary optimization metric, histogram gradient boosting is currently the preferred model. However, its recall, precision, and F1-score must also be examined before making the final selection.

In [9]:
tuned_histogram_scores = cross_validate(
    estimator=histogram_search.best_estimator_,
    X=X_train,
    y=y_train,
    cv=cross_validation,
    scoring=scoring,
    n_jobs=-1
)

tuned_histogram_summary = pd.Series({
    metric: tuned_histogram_scores[f"test_{metric}"].mean()
    for metric in scoring
})

tuned_model_comparison = pd.concat(
    [
        tuned_logistic_summary.rename("Tuned logistic regression"),
        tuned_histogram_summary.rename("Tuned histogram boosting")
    ],
    axis=1
)

display(tuned_model_comparison.round(4))

,Tuned logistic regression,Tuned histogram boosting
accuracy,0.8927,0.8940
precision,0.6643,0.6220
recall,0.1676,0.2392
f1,0.2675,0.3453
roc_auc,0.7633,0.7980
pr_auc,0.4025,0.4488


### Final model selection

The tuned histogram gradient boosting model outperforms the tuned logistic regression model on accuracy, recall, F1-score, ROC-AUC, and PR-AUC.

The logistic regression model has slightly higher precision, meaning its positive predictions are somewhat more reliable. However, it identifies substantially fewer actual subscribers.

Histogram gradient boosting provides the stronger overall balance between ranking quality, minority-class detection, and predictive performance.

Therefore, the tuned histogram gradient boosting model is selected as the final model candidate.